In [3]:
%pip install -q pandas numpy scikit-learn xgboost
print("Dependencies installed.")

Note: you may need to restart the kernel to use updated packages.
Dependencies installed.


In [13]:
import warnings
warnings.filterwarnings("ignore")

In [14]:
import pandas as pd
import numpy as np

# hiển thị toàn bộ cột khi in dataframe
pd.set_option('display.max_columns', None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [15]:
file_path = r".\data\suumo_raw_data.csv"


print("File path set:", file_path)

File path set: .\data\suumo_raw_data.csv


In [16]:
def load_dataset(file_path):

    try:
        df = pd.read_csv(file_path, encoding="utf-8")
    except:
        df = pd.read_csv(file_path, encoding="cp932")

    return df

In [17]:
file_path = r".\data\suumo_raw_data.csv"

df = pd.read_csv(file_path)

print(df.columns)

Index(['id', '電話番号', '家賃', '管理費_共益費', '敷金', '礼金', '保証金', '敷引_償却', '所在地', '駅徒歩',
       '間取り', '専有面積', '築年数', '階', '向き', '建物種別', '間取り詳細', '構造', '階建', '築年月',
       'エネルギー消費性能', '断熱性能', '目安光熱費', '損保', '駐車場', '入居', '取引態様', '条件',
       '取り扱い店舗物件コード', 'SUUMO物件コード', '総戸数', '情報更新日', '次回更新日', '契約期間', '保証会社',
       'ほか諸費用', '備考', 'create_at'],
      dtype='str')


In [18]:
def clean_money(col):

    col = col.replace("-", np.nan)
    col = col.astype(str)

    col = col.str.replace(",", "", regex=False)
    col = col.str.replace("万円", "", regex=False)
    col = col.str.replace("円", "", regex=False)

    col = col.str.extract(r'(\d+\.?\d*)')[0]

    return col.astype(float)

In [19]:
print("Columns in dataset:")
print(df.columns)

Columns in dataset:
Index(['id', '電話番号', '家賃', '管理費_共益費', '敷金', '礼金', '保証金', '敷引_償却', '所在地', '駅徒歩',
       '間取り', '専有面積', '築年数', '階', '向き', '建物種別', '間取り詳細', '構造', '階建', '築年月',
       'エネルギー消費性能', '断熱性能', '目安光熱費', '損保', '駐車場', '入居', '取引態様', '条件',
       '取り扱い店舗物件コード', 'SUUMO物件コード', '総戸数', '情報更新日', '次回更新日', '契約期間', '保証会社',
       'ほか諸費用', '備考', 'create_at'],
      dtype='str')


In [20]:
df = load_dataset(file_path)

df.rename(columns={
    "area": "area_m2"
}, inplace=True)

In [21]:
df.rename(columns={
    "専有面積": "area_m2"
}, inplace=True)

In [22]:
def clean_dataset(file_path):

    df = load_dataset(file_path)

    # Canonical English schema for this dataset (38 columns)
    english_columns = [
        "id",
        "phone_number",
        "rent",
        "management_fee",
        "deposit",
        "key_money",
        "guarantee_deposit",
        "deposit_amortization",
        "location",
        "walking_distance_to_station",
        "layout",
        "area_m2",
        "building_age",
        "floor_raw",
        "direction",
        "building_type",
        "layout_details",
        "structure",
        "stories",
        "built_date",
        "energy_efficiency",
        "insulation_performance",
        "estimated_utility_cost",
        "insurance",
        "parking",
        "move_in",
        "transaction_type",
        "conditions",
        "agency_property_code",
        "suumo_property_code",
        "total_units",
        "last_updated_date",
        "next_update_date",
        "contract_period",
        "guarantor_company",
        "other_fees",
        "remarks",
        "created_at",
    ]

    if len(df.columns) == len(english_columns):
        df.columns = english_columns
    else:
        # Fallback for already-partial English data
        df.rename(columns={
            "create_at": "created_at",
            "area": "area_m2",
            "age": "building_age",
            "distance": "walking_distance_to_station",
        }, inplace=True)

    money_columns = [
        "rent",
        "management_fee",
        "deposit",
        "key_money",
        "guarantee_deposit",
        "deposit_amortization",
        "estimated_utility_cost",
        "insurance",
        "other_fees",
    ]

    for col in money_columns:
        if col in df.columns:
            df[col] = clean_money(df[col])

    if "area_m2" in df.columns:
        df["area_m2"] = (
            df["area_m2"]
            .astype(str)
            .str.extract(r"(\d+\.?\d*)")[0]
            .astype(float)
        )

    return df


In [23]:
file_path = r".\data\suumo_raw_data.csv"

df = clean_dataset(file_path)

df.to_csv(r".\data\yodogawa_data_cleaning.csv", index=False)